### 03. Data Preprocessing - Automobile Loan Default Prediction


## 1. Setup



In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Relative path handling: works whether launched from the workspace root or inside the notebooks folder
data_path = 'data/raw/Train_Dataset.csv' if os.path.exists('data/raw/Train_Dataset.csv') else '../data/raw/Train_Dataset.csv'

df = pd.read_csv(data_path, low_memory=False)

print(f"Dataset loaded successfully from: '{data_path}'")
print(f"Shape: {df.shape}")

Dataset loaded successfully from: '../data/raw/Train_Dataset.csv'
Shape: (121856, 40)


## 2. Fix Data Types

Convert numeric-looking text columns (found in `01_data_understanding.ipynb`) to real numbers; invalid entries become missing.


In [3]:
candidate_numerical_cols = [
    'Client_Income', 'Credit_Amount', 'Loan_Annuity', 'Population_Region_Relative',
    'Age_Days', 'Employed_Days', 'Registration_Days', 'ID_Days', 'Score_Source_3'
]

for col in candidate_numerical_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[candidate_numerical_cols].dtypes)

Client_Income                 float64
Credit_Amount                 float64
Loan_Annuity                  float64
Population_Region_Relative    float64
Age_Days                      float64
Employed_Days                 float64
Registration_Days             float64
ID_Days                       float64
Score_Source_3                float64
dtype: object


## 3. Remove Duplicate Rows

`ID` is unique per row, so a duplicate check that includes it can never find a match. Ignoring `ID`, 2,640 rows are exact copies of another row (per the team implementation plan) - remove them now, before the split, so copies can't land on both sides.


In [4]:
cols_no_id = [c for c in df.columns if c != 'ID']
before_rows = df.shape[0]
df = df.drop_duplicates(subset=cols_no_id).reset_index(drop=True)
after_rows = df.shape[0]

print(f"Removed {before_rows - after_rows} duplicate rows (ignoring ID)")
print("Shape after deduplication:", df.shape)

Removed 2640 duplicate rows (ignoring ID)
Shape after deduplication: (119216, 40)


## 4. Rule-Based Placeholder/Sentinel Fixes

Fixed, non-data-dependent fixes found in `02_eda.ipynb` and the team implementation plan.

`Own_House_Age` is actually a car-age column, not a house-age column: it is filled only when `Car_Owned == 1`, unrelated to `House_Own`. Rename it to `car_age` and add an explicit `has_car_age` indicator, replacing the generic missing-flag for this column.


In [5]:
# Employed_Days sentinel -> flag + missing
df['Is_Retired_Or_Unemployed'] = (df['Employed_Days'] == 365243).astype(int)
df.loc[df['Employed_Days'] == 365243, 'Employed_Days'] = np.nan

# Disguised missing values -> NaN (Type_Organization's "XNA" is kept as-is, it's a valid category)
df.loc[df['Client_Gender'] == 'XNA', 'Client_Gender'] = np.nan
df.loc[df['Accompany_Client'] == '##', 'Accompany_Client'] = np.nan

# Corrupted numeric values -> NaN
df.loc[df['Score_Source_2'] > 1, 'Score_Source_2'] = np.nan
df.loc[df['Population_Region_Relative'] > 1, 'Population_Region_Relative'] = np.nan

# Own_House_Age is actually car age (filled only when Car_Owned == 1), not house age
verified_car_age_link = (df['Own_House_Age'].notna() == (df['Car_Owned'] == 1)).mean()
print(f"Own_House_Age presence matches Car_Owned==1 for {verified_car_age_link:.4%} of rows")

df['has_car_age'] = (df['Car_Owned'] == 1).astype(int)
df = df.rename(columns={'Own_House_Age': 'car_age'})

# Drop non-predictive identifier
df = df.drop(columns=['ID'])

print("Is_Retired_Or_Unemployed counts:")
print(df['Is_Retired_Or_Unemployed'].value_counts())
print()
print("Shape after this step:", df.shape)

Own_House_Age presence matches Car_Owned==1 for 98.9817% of rows
Is_Retired_Or_Unemployed counts:
Is_Retired_Or_Unemployed
0    98609
1    20607
Name: count, dtype: int64

Shape after this step: (119216, 41)


## 5. Stratified Train/Test Split

Split now, before any step that learns from the data, to avoid leakage. Stratify on `Default` to preserve the ~91.9%/8.1% class balance.


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['Default'], random_state=42
)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print()
print("Default rate (train):", train_df['Default'].mean().round(4))
print("Default rate (test):", test_df['Default'].mean().round(4))

train_df shape: (95372, 41)
test_df shape: (23844, 41)

Default rate (train): 0.081
Default rate (test): 0.081


## 6. Handle Missing Values

Rule: numeric -> median (fit on train), with a `_was_missing` flag added when missing rate > 5% (preserves signal for high-missing columns). Categorical -> fill with `"Missing"` category (keeps informative missingness, e.g. `Client_Occupation`). Exception: `car_age` filled with 0 (no car owned), not median, and excluded from the generic flag rule since `has_car_age` (Step 4) already serves that purpose.


In [7]:
numeric_cols_missing = [c for c in train_df.select_dtypes(include=[np.number]).columns
                        if c not in ['Default', 'has_car_age'] and train_df[c].isna().sum() > 0]
categorical_cols_missing = [c for c in train_df.select_dtypes(include=['object', 'string']).columns
                            if train_df[c].isna().sum() > 0]

flag_threshold = 0.05
missing_value_fills = {}
flags_added = []

for col in numeric_cols_missing:
    if col != 'car_age' and train_df[col].isna().mean() > flag_threshold:
        flag_col = col + '_was_missing'
        train_df[flag_col] = train_df[col].isna().astype(int)
        test_df[flag_col] = test_df[col].isna().astype(int)
        flags_added.append(flag_col)

    fill_value = 0 if col == 'car_age' else train_df[col].median()
    missing_value_fills[col] = fill_value
    train_df[col] = train_df[col].fillna(fill_value)
    test_df[col] = test_df[col].fillna(fill_value)

for col in categorical_cols_missing:
    train_df[col] = train_df[col].fillna('Missing')
    test_df[col] = test_df[col].fillna('Missing')

print("Missing-indicator flags added:", flags_added)
print()
print("Remaining missing values (train):", train_df.isnull().sum().sum())
print("Remaining missing values (test):", test_df.isnull().sum().sum())

Missing-indicator flags added: ['Employed_Days_was_missing', 'ID_Days_was_missing', 'Score_Source_1_was_missing', 'Score_Source_3_was_missing', 'Social_Circle_Default_was_missing', 'Credit_Bureau_was_missing']

Remaining missing values (train): 0
Remaining missing values (test): 0


## 7. Outlier Treatment

Log-transform `Client_Income` (extreme skew found in EDA). `Credit_Amount`/`Loan_Annuity` left as-is (moderate skew, not extreme; tree-based models unaffected).


In [8]:
print("Client_Income skew before:", train_df['Client_Income'].skew().round(2))

train_df['Client_Income'] = np.log1p(train_df['Client_Income'])
test_df['Client_Income'] = np.log1p(test_df['Client_Income'])

print("Client_Income skew after:", train_df['Client_Income'].skew().round(2))

Client_Income skew before: 42.86


Client_Income skew after: 0.18


## 8. Encode Categorical Variables

One-hot encode categorical columns except those with small-sample categories needing grouping first (`Type_Organization`, `Client_Education`, `Client_Income_Type`, `Client_Occupation`) - deferred to `04_feature_engineering.ipynb`. Fit on train, align test to the same columns.


In [9]:
categorical_cols_to_encode = [c for c in train_df.select_dtypes(include=['object', 'string']).columns
                              if c not in ['Type_Organization', 'Client_Education', 'Client_Income_Type', 'Client_Occupation']]

train_df = pd.get_dummies(train_df, columns=categorical_cols_to_encode, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols_to_encode, drop_first=True)

train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)

dummy_cols = [c for c in train_df.columns if train_df[c].dtype == bool]
train_df[dummy_cols] = train_df[dummy_cols].astype(int)
test_df[dummy_cols] = test_df[dummy_cols].astype(int)

print("Encoded columns:", categorical_cols_to_encode)
print("Left unencoded for 04_feature_engineering.ipynb:", ['Type_Organization', 'Client_Education', 'Client_Income_Type', 'Client_Occupation'])
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

Encoded columns: ['Accompany_Client', 'Client_Marital_Status', 'Client_Gender', 'Loan_Contract_Type', 'Client_Housing_Type', 'Client_Permanent_Match_Tag', 'Client_Contact_Work_Tag']
Left unencoded for 04_feature_engineering.ipynb: ['Type_Organization', 'Client_Education', 'Client_Income_Type', 'Client_Occupation']
train_df shape: (95372, 62)
test_df shape: (23844, 62)


## 9. Save Processed Data

Save train/test sets for `04_feature_engineering.ipynb`. Scaling is deferred to the end of `04_feature_engineering.ipynb`, after feature engineering, so newly derived features get scaled too.


In [10]:
processed_dir = 'data/processed' if os.path.exists('data/processed') else '../data/processed'

train_df.to_csv(os.path.join(processed_dir, 'train_processed.csv'), index=False)
test_df.to_csv(os.path.join(processed_dir, 'test_processed.csv'), index=False)

print(f"Saved train_processed.csv: {train_df.shape}")
print(f"Saved test_processed.csv: {test_df.shape}")

Saved train_processed.csv: (95372, 62)
Saved test_processed.csv: (23844, 62)


## 10. Summary

| Step | Decision |
|---|---|
| Data types | 9 text columns converted to numeric |
| Duplicates | 2,640 exact duplicate rows (ignoring `ID`) removed before split |
| Sentinel/placeholders | `Employed_Days` sentinel -> flag + missing; `XNA`/`##` -> missing; `Score_Source_2`/`Population_Region_Relative` >1 -> missing; `ID` dropped |
| Split | Stratified 80/20 on `Default` (8.10% both sides, deduplicated) |
| Missing values | Numeric -> median + `_was_missing` flag if >5% missing; categorical -> `"Missing"` category; `Own_House_Age` -> 0 |
| Outliers | `Client_Income` log1p (skew 41.5 -> 0.19) |
| Encoding | One-hot for low-cardinality categorical columns; `Type_Organization`, `Client_Education`, `Client_Income_Type`, `Client_Occupation` left for `04_feature_engineering.ipynb` (rare-category grouping needed first) |
| Scaling | Deferred to `04_feature_engineering.ipynb` (after feature engineering) |
| Output | `train_processed.csv`, `test_processed.csv` in `data/processed/` |

Next: `04_feature_engineering.ipynb` - group rare categories, derive features, scale, feature selection.
